# Hierarchy saliency through the Python API

All saliency, validation, ranking, extinction, and contour logic runs in the C++ library exposed by `mmcfilters`. Python prepares inputs, invokes bindings, and formats tables and figures.

A formal `HierarchySaliencyMap` result is edge-indexed. Edge figures draw those values directly. `edgeMapToPixelImage(...)` is a C++ display projection and does not replace the formal map. Primary reference: Jean Cousty, Laurent Najman, Yukiko Kenmochi, and Silvio Guimarães, [*Hierarchical segmentations with graphs: quasi-flat zones, minimum spanning trees, and saliency maps*](https://doi.org/10.1007/s10851-017-0768-7), *Journal of Mathematical Imaging and Vision* 60(4), 479–502, 2018. Complete references and the code-to-paper correspondence are in [`docs/saliency.md`](../docs/saliency.md#primary-references-and-implementation-correspondence).

## API contract

- `validateHierarchyValuation(...)` validates dense node valuations in C++.
- `rankHierarchyValuation(...)` ranks hierarchy levels in C++.
- `computeSaliencyEdgeMap(...)` performs LCA projection and writes zero on same-owner edges.
- `getExtinctionValueAttribute()` exposes the max-descendant extinction valuation.
- `computeFormalSaliencyEdgeMap(...)` runs Cousty persistence, not direct LCA projection.
- `computeMonotoneExtinctionProjection(...)` explicitly provides the former LCA behavior.
- `thresholdCut(...)` and `edgeMapToPixelImage(...)` run in C++. The latter is visualization only.

In [ ]:
from __future__ import annotations

import mmcfilters
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
import numpy as np
from skimage import data

%matplotlib inline
print(f"mmcfilters: {mmcfilters.__version__}")


In [ ]:
CROP = None  # Example: (slice(64, 224), slice(64, 224))
RADIUS = 1.5

image = np.ascontiguousarray(data.camera(), dtype=np.uint8)
if CROP is not None:
    image = np.ascontiguousarray(image[CROP])
print(image.shape, image.dtype, int(image.min()), int(image.max()))

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray")
plt.title("skimage.data.camera()")
plt.axis("off");


In [ ]:
max_tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(image, radius=RADIUS)
min_tree = mmcfilters.MorphologicalTreeFactory.createMinTree(image, radius=RADIUS)

print("max-tree nodes:", max_tree.numInternalNodeSlots, "alive:", len(max_tree.aliveNodeIds))
print("min-tree nodes:", min_tree.numInternalNodeSlots, "alive:", len(min_tree.aliveNodeIds))


In [ ]:
# Only output formatting: these helpers do not compute saliency decisions.
def summarize_edge_map(name: str, edge_map: dict) -> None:
    values = np.asarray(edge_map["values"])
    print(
        f"{name:30s} edges={values.size:7d} "
        f"dtype={values.dtype!s:8s} min={values.min():10.4g} "
        f"max={values.max():10.4g} mean={values.mean():10.4g}"
    )


def edge_segments(edge_map: dict) -> np.ndarray:
    rows = int(edge_map["numRows"])
    cols = int(edge_map["numCols"])
    sources = np.asarray(edge_map["sources"], dtype=np.int64)
    targets = np.asarray(edge_map["targets"], dtype=np.int64)
    source_y, source_x = np.divmod(sources, cols)
    target_y, target_x = np.divmod(targets, cols)
    return np.stack(
        [
            np.column_stack([source_x, source_y]),
            np.column_stack([target_x, target_y]),
        ],
        axis=1,
    )


def plot_edge_maps(image_uint8: np.ndarray, maps: list[tuple[str, dict]], linewidth: float = 0.35) -> None:
    fig, axes = plt.subplots(1, len(maps) + 1, figsize=(4 * (len(maps) + 1), 4), constrained_layout=True)
    axes[0].imshow(image_uint8, cmap="gray")
    axes[0].set_title("image")
    axes[0].axis("off")
    for ax, (title, edge_map) in zip(axes[1:], maps):
        values = np.asarray(edge_map["values"], dtype=np.float64)
        ax.imshow(image_uint8, cmap="gray", alpha=0.25)
        collection = LineCollection(edge_segments(edge_map), array=values, cmap="gray", linewidths=linewidth)
        ax.add_collection(collection)
        ax.set_xlim(-0.5, image_uint8.shape[1] - 0.5)
        ax.set_ylim(image_uint8.shape[0] - 0.5, -0.5)
        ax.set_aspect("equal")
        ax.set_title(title)
        ax.axis("off")
        fig.colorbar(collection, ax=ax, fraction=0.046, pad=0.04)
    plt.show()


def contour_as_edge_map(contour: dict, value: float = 1.0) -> dict:
    out = dict(contour)
    out["values"] = np.full(np.asarray(contour["sources"]).shape[0], value, dtype=np.float64)
    return out


In [ ]:
topological_edge_map = mmcfilters.HierarchySaliencyMap.computeTopologicalLevelEdgeMap(max_tree)
normalized_altitude_edge_map = mmcfilters.HierarchySaliencyMap.computeNormalizedAltitudeEdgeMap(max_tree)

area = mmcfilters.Attribute.computeSingleTopologyAttribute(max_tree, mmcfilters.Attribute.AREA).astype(np.float32)
mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(max_tree, area, nonnegative=True)
area_edge_map = mmcfilters.HierarchySaliencyMap.computeSaliencyEdgeMap(max_tree, area)

ranked_area = mmcfilters.HierarchySaliencyMapValidation.rankHierarchyValuation(max_tree, area)
ranked_area_edge_map = mmcfilters.HierarchySaliencyMap.computeSaliencyEdgeMap(max_tree, ranked_area)

extinction = mmcfilters.ExtinctionValues(max_tree, area)
ranked_extinction_valuation = extinction.computeRankedExtinctionValueAttribute()
mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(max_tree, ranked_extinction_valuation, nonnegative=True)
ranked_extinction_edge_map = extinction.computeFormalSaliencyEdgeMap(ranked=True)

min_normalized_altitude_edge_map = mmcfilters.HierarchySaliencyMap.computeNormalizedAltitudeEdgeMap(min_tree)



edge_maps = {
    "topological": topological_edge_map,
    "normalized altitude": normalized_altitude_edge_map,
    "area": area_edge_map,
    "ranked area": ranked_area_edge_map,
    "ranked extinction": ranked_extinction_edge_map,
    "min normalized altitude": min_normalized_altitude_edge_map,
}

for name, edge_map in edge_maps.items():
    summarize_edge_map(name, edge_map)


In [ ]:
topological_max_projection = mmcfilters.HierarchySaliencyMapProjection.edgeMapToPixelImage(
    ranked_extinction_edge_map,
    mmcfilters.EdgeToPixelReducer.Mean,
)

fig, axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
axes[0].imshow(image, cmap="gray")
axes[0].set_title("image")
axes[0].axis("off")

im = axes[1].imshow(topological_max_projection, cmap="gray")
axes[1].set_title("topologico - projecao")
axes[1].axis("off")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.show()


In [ ]:
plot_edge_maps(
    image,
    [
        ("topologico", topological_edge_map),
        ("normalized altitude", normalized_altitude_edge_map),
        ("area rankeada", ranked_area_edge_map),
        ("ranked extinction persistence", ranked_extinction_edge_map),
    ],
)


In [ ]:
CONTOUR_THRESHOLDS = [0.5, 0.75, 0.99]

normalized_cuts = []
for threshold in CONTOUR_THRESHOLDS:
    cut = mmcfilters.HierarchySaliencyMapProjection.thresholdCut(normalized_altitude_edge_map, threshold)
    normalized_cuts.append((f"lambda >= {threshold:.2f}", contour_as_edge_map(cut)))
    print(f"normalized lambda={threshold:.2f}: {len(cut['sources']):7d} edges")

rank_thresholds = [1, 2, 3]
extinction_cuts = []
for threshold in rank_thresholds:
    cut = mmcfilters.HierarchySaliencyMapProjection.thresholdCut(ranked_extinction_edge_map, threshold)
    extinction_cuts.append((f"rank >= {threshold}", contour_as_edge_map(cut)))
    print(f"extinction rank>={threshold}: {len(cut['sources']):7d} edges")


In [ ]:
plot_edge_maps(image, normalized_cuts, linewidth=0.45)
plot_edge_maps(image, extinction_cuts, linewidth=0.45)


## Notes

This notebook keeps algorithmic decisions in the library. Invalid valuations must fail through `validateHierarchyValuation`. Formal maps come from `computeSaliencyEdgeMap` or the persistence-based `ExtinctionValues.computeFormalSaliencyEdgeMap`.

`plot_edge_maps` only draws returned edges; it does not create or modify saliency values.